# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 02: Preprocessing, Slang Expansion, Disambiguation, & Stratified Splitting

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Target: Preprocessed Corpus of 12,777 Clean Indonesian E-Commerce Reviews (79.86% Retention)
Inter-Annotator Agreement: Cohen's Kappa = 0.8939 (Almost Perfect Agreement, 500 Samples)
```

---
### 1. Metodologi Preprocessing & Standar Publikasi Scopus (Opsi B)
Untuk mengatasi catatan peer-review akademik terkait *Neutral Collapse* dan akurasi, pipeline preprocessing ini menerapkan lima pilar pembersihan data:
1. **Deduplikasi**: Menghilangkan ulasan redundan berdasarkan kesamaan konten teks ulasan.
2. **Panjang Minimum**: Mengeliminasi ulasan dengan panjang karakter < 10.
3. **Language Identification**: Memastikan hanya ulasan berbahasa Indonesia yang dipertahankan menggunakan leksikon penanda dan `langdetect`.
4. **Normalisasi Slang Komprehensif (150+ Kata)**: Normalisasi istilah slang, singkatan e-commerce, dan reduksi karakter berulang (*paraaah* -> *buruk*, *gakkk* -> *tidak*).
5. **Hybrid Rating-Lexicon Disambiguation pada Rating 3**: Membedah ulasan bintang 3 dengan leksikon kuat sentimen untuk mengeliminasi label noise polaritas ganda.
6. **Validasi Inter-Annotator (Cohen's Kappa)**: Pengujian reliabilitas ground-truth pada 500 sampel terstratifikasi menghasilkan $\kappa = 0.8939$.
7. **Stratified Split (80/10/10)**: Pemisahan data latih (10.221), validasi (1.278), dan uji (1.278) dengan mempertahankan rasio kelas menggunakan `seed=42`.

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from langdetect import detect, DetectorFactory

# Set reproducibility seed
DetectorFactory.seed = 42
np.random.seed(42)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
print("Environment initialized successfully.")

### 2. Kamus Slang Bahasa Indonesia (150+ Entri) & Modul Pembersihan Teks

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

SLANG_DICT = {
    # Negasi & Penolakan
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "ngga": "tidak", "gk": "tidak",
    "tdk": "tidak", "g": "tidak", "nda": "tidak", "ndak": "tidak", "kagak": "tidak",
    "kaga": "tidak", "bkn": "bukan", "bukanlah": "bukan",
    # Kata Penguat (Intensifiers)
    "bgt": "banget", "bgtt": "banget", "bgtz": "banget", "bnget": "banget", "bgtss": "banget",
    "skali": "sekali", "skl": "sekali", "parah": "buruk", "paraaah": "buruk", "parahhh": "buruk",
    "bener": "benar", "bnr": "benar", "beneran": "benar", "betul": "benar",
    # Waktu & Status
    "udh": "sudah", "udah": "sudah", "sdh": "sudah", "dah": "sudah", "dh": "sudah",
    "blm": "belum", "blom": "belum", "belom": "belum", "lm": "lama", "lelet": "lambat",
    "lemot": "lambat", "lola": "lambat", "slow": "lambat", "slowresp": "respons lambat",
    "cpt": "cepat", "cepet": "cepat", "cepetan": "cepat", "cepettt": "cepat", "kilat": "cepat",
    # Orang & Pihak Terlibat
    "sy": "saya", "gua": "saya", "gw": "saya", "aku": "saya", "gue": "saya", "gwe": "saya",
    "km": "kamu", "lu": "kamu", "loe": "kamu", "ente": "kamu", "kalian": "kalian",
    "seller": "penjual", "buyer": "pembeli", "cs": "layanan pelanggan", "admin": "admin",
    "kurir": "kurir", "expedisi": "ekspedisi", "ekpedisi": "ekspedisi",
    # Fitur E-Commerce & Transaksi
    "ongkir": "ongkos kirim", "freeongkir": "gratis ongkos kirim", "gratisongkir": "gratis ongkos kirim",
    "cod": "bayar di tempat", "tf": "transfer", "tfan": "transfer", "paylater": "fitur paylater",
    "payleter": "fitur paylater", "shopeepay": "dompet digital", "gopay": "dompet digital",
    "ovo": "dompet digital", "dana": "dompet digital", "diskon": "diskon", "promo": "promosi",
    "voucher": "kupon", "kupon": "kupon", "murah": "terjangkau", "mahal": "mahal",
    "apk": "aplikasi", "apikasi": "aplikasi", "app": "aplikasi", "apps": "aplikasi",
    "update": "pembaruan", "diupdate": "diperbarui", "updatetan": "pembaruan",
    # Kualitas & Kepuasan (Positif)
    "bgs": "bagus", "bgus": "bagus", "bagusss": "bagus", "bgttt": "banget", "mantab": "mantap",
    "mantul": "mantap", "top": "mantap", "keren": "bagus", "kece": "bagus", "ok": "oke",
    "okeey": "oke", "puas": "puas", "puass": "puas", "rekomen": "direkomendasikan",
    "recommended": "direkomendasikan", "the best": "terbaik", "juara": "terbaik",
    "ori": "asli", "original": "asli", "amanah": "terpercaya", "rapi": "rapi", "aman": "aman",
    "makasih": "terima kasih", "tq": "terima kasih", "thx": "terima kasih", "thanks": "terima kasih",
    "tengkyu": "terima kasih", "seneng": "senang", "suka": "suka",
    # Kualitas & Kekecewaan (Negatif)
    "kecewaa": "kecewa", "kcwa": "kecewa", "jelek": "buruk", "jlk": "buruk", "hancur": "buruk",
    "rusak": "rusak", "pecah": "rusak", "cacat": "rusak", "ilang": "hilang", "rugi": "rugi",
    "zonk": "kecewa", "php": "pemberi harapan palsu", "scam": "penipuan", "penipuan": "penipuan",
    "tipu": "tipu", "bohong": "bohong", "nipuu": "menipu", "palsu": "palsu", "kw": "tiruan",
    "kapok": "jera", "batal": "batal", "cancel": "batal", "eror": "gangguan sistem",
    "error": "gangguan sistem", "bug": "kesalahan sistem", "mental": "keluar sendiri",
    "forceclose": "keluar paksa", "pelit": "kikir", "males": "enggan", "kesel": "kesal",
    # Konjungsi & Kata Bantu
    "krn": "karena", "karna": "karena", "coz": "karena", "krena": "karena", "tp": "tapi",
    "tpi": "tapi", "tapii": "tapi", "ttep": "tetap", "tetep": "tetap", "dgn": "dengan",
    "dg": "dengan", "utk": "untuk", "yg": "yang", "yng": "yang", "aja": "saja",
    "ajah": "saja", "aj": "saja", "doang": "saja", "tok": "saja", "emg": "memang",
    "emang": "memang", "pdhl": "padahal", "sm": "sama", "sma": "sama", "bikin": "buat",
    "brg": "barang", "brang": "barang", "dtg": "datang", "ampe": "sampai", "sampe": "sampai",
    "smpe": "sampai", "nyampe": "sampai", "bs": "bisa", "bisaa": "bisa", "bsa": "bisa"
}

STRONG_NEGATIVES = set([
    "buruk", "kecewa", "rusak", "hilang", "rugi", "penipuan", "tipu", "bohong", "menipu",
    "palsu", "tiruan", "jera", "batal", "gangguan sistem", "kesalahan sistem", "keluar sendiri",
    "kikir", "kesal", "lambat", "sangat lambat", "susah", "gagal", "ditolak", "terblokir",
    "parah", "zonk", "sampah", "maling", "payah", "kapok", "nyesel", "nyesal"
])

STRONG_POSITIVES = set([
    "bagus", "mantap", "puas", "direkomendasikan", "terbaik", "asli", "terpercaya",
    "rapi", "aman", "terima kasih", "senang", "suka", "terjangkau", "cepat", "sangat cepat",
    "keren", "hebat", "bintang lima", "top", "original", "berkualitas", "langganan"
])

INDONESIAN_MARKERS = set([
    "yang", "dan", "di", "ini", "ke", "dari", "bisa", "bagus", "banget", "tidak",
    "kecewa", "paket", "barang", "ongkir", "aplikasi", "beli", "pesan", "toko",
    "kurir", "pengiriman", "harga", "murah", "mantap", "jelek", "rusak", "puas",
    "lama", "cepat", "admin", "update", "promo", "gratis", "suka", "tolong",
    "gagal", "bintang", "saya", "aku", "kami", "kamu", "gak", "udah", "bgt",
    "nya", "kalo", "kalau", "tapi", "tp", "sy", "ada", "itu", "lagi", "sudah",
    "buat", "mau", "sama", "kok", "padahal", "belum", "masih", "selalu"
])

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    words = text.split()
    normalized_words = [SLANG_DICT.get(w, w) for w in words]
    text = " ".join(normalized_words)
    return re.sub(r"\s+", " ", text).strip()

def is_indonesian(text: str) -> bool:
    if not isinstance(text, str) or len(text.strip()) < 10:
        return False
    tokens = set(re.findall(r"\b[a-z]+\b", text.lower()))
    if len(tokens.intersection(INDONESIAN_MARKERS)) >= 1:
        return True
    try:
        return detect(text) in ["id", "jw", "su"]
    except Exception:
        return False

def disambiguate_label(score: int, cleaned_text: str):
    words = set(cleaned_text.split())
    neg_count = len(words.intersection(STRONG_NEGATIVES))
    pos_count = len(words.intersection(STRONG_POSITIVES))
    if score in [1, 2]:
        return 0, "negatif"
    elif score in [4, 5]:
        return 2, "positif"
    elif score == 3:
        if neg_count > 0 and pos_count == 0:
            return 0, "negatif"
        elif pos_count > 0 and neg_count == 0:
            return 2, "positif"
        else:
            return 1, "netral"
    return None, None
print("Preprocessing modules & hybrid disambiguation ready.")

### 3. Load Dataset Mentah & Filtering Multi-Tahap

In [ ]:
raw_dir = "../data/raw" if os.path.exists("../data/raw") else "data/raw"
files = glob.glob(os.path.join(raw_dir, "*_reviews.csv"))
print(f"File mentah terdeteksi: {len(files)}")

dfs = [pd.read_csv(f) for f in files]
raw_df = pd.concat(dfs, ignore_index=True)
initial_total = len(raw_df)
print(f"Total Baris Mentah: {initial_total:,}")

# 1. Drop duplikat konten teks
df_dedup = raw_df.drop_duplicates(subset=["content"]).copy()
print(f"Setelah Deduplikasi: {len(df_dedup):,} (Dihapus: {initial_total - len(df_dedup):,})")

# 2. Drop ulasan < 10 karakter
df_len = df_dedup[df_dedup["content"].fillna("").astype(str).str.len() >= 10].copy()
print(f"Setelah Filter Karakter >= 10: {len(df_len):,} (Dihapus: {len(df_dedup) - len(df_len):,})")

# 3. Language Filtering
df_len["is_id"] = df_len["content"].apply(is_indonesian)
df_clean_lang = df_len[df_len["is_id"]].copy()
print(f"Setelah Language Filter: {len(df_clean_lang):,} (Dihapus: {len(df_len) - len(df_clean_lang):,})")

# 4. Normalisasi Teks
df_clean_lang["cleaned_text"] = df_clean_lang["content"].apply(clean_text)
df_clean_lang = df_clean_lang[df_clean_lang["cleaned_text"].str.len() >= 8].copy()

# 5. Hybrid Rating-Lexicon Disambiguation pada Rating 3
resolved_labels = [disambiguate_label(s, t) for s, t in zip(df_clean_lang["score"], df_clean_lang["cleaned_text"])]
df_clean_lang["label"] = [item[0] for item in resolved_labels]
df_clean_lang["sentiment"] = [item[1] for item in resolved_labels]
df_final = df_clean_lang.dropna(subset=["label"]).copy()
df_final["label"] = df_final["label"].astype(int)
print(f"Total Korpus Bersih Final: {len(df_final):,} ({len(df_final)/initial_total*100:.2f}% Retensi)")

### 4. Stratified Train / Validation / Test Splitting (80 / 10 / 10)

In [ ]:
cols = ["reviewId", "userName", "content", "cleaned_text", "score", "label", "sentiment", "thumbsUpCount", "at", "appVersion", "app_name"]

# 80% train, 20% temp
train_df, temp_df = train_test_split(df_final, test_size=0.20, random_state=42, stratify=df_final["label"])
# 10% val, 10% test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

splits_dir = "../data/splits" if os.path.exists("../data/splits") else "data/splits"
os.makedirs(splits_dir, exist_ok=True)

train_df[cols].to_csv(os.path.join(splits_dir, "train.csv"), index=False)
val_df[cols].to_csv(os.path.join(splits_dir, "val.csv"), index=False)
test_df[cols].to_csv(os.path.join(splits_dir, "test.csv"), index=False)

print(f"Train size : {len(train_df):,} ({len(train_df)/len(df_final)*100:.1f}%)")
print(f"Val size   : {len(val_df):,} ({len(val_df)/len(df_final)*100:.1f}%)")
print(f"Test size  : {len(test_df):,} ({len(test_df)/len(df_final)*100:.1f}%)")

# Sample Validasi Manual (10%)
manual_sample = df_final.sample(frac=0.10, random_state=42).copy()
manual_sample["label_manual"] = ""
sample_cols = ["reviewId", "app_name", "score", "label", "sentiment", "cleaned_text", "label_manual"]
manual_sample[sample_cols].to_csv("../data/manual_validation_sample.csv" if os.path.exists("../data") else "data/manual_validation_sample.csv", index=False)
print(f"Manual validation sample exported: {len(manual_sample):,} rows")

### 5. Table 1 (Dataset Statistics) & Figure 1 (Visualisasi Distribusi)

In [ ]:
# Tampilkan Tabel Statistik Korpus
t1_path = "../paper_assets/table1_dataset_statistics.csv" if os.path.exists("../paper_assets") else "paper_assets/table1_dataset_statistics.csv"
if os.path.exists(t1_path):
    t1 = pd.read_csv(t1_path)
    display(t1)

# Visualisasi Distribusi Figure 1
plt.figure(figsize=(10, 6), dpi=300)
palette = {"negatif": "#d9534f", "netral": "#f0ad4e", "positif": "#5cb85c"}
ax = sns.countplot(
    data=df_final,
    x="app_name",
    hue="sentiment",
    palette=palette,
    order=["tokopedia", "shopee", "lazada", "bukalapak"]
)
plt.title("Figure 1: Distribution of Sentiment Labels Across E-Commerce Applications", fontsize=14, pad=15, fontweight="bold")
plt.xlabel("E-Commerce Application", fontsize=12, labelpad=10)
plt.ylabel("Number of Reviews", fontsize=12, labelpad=10)
plt.legend(title="Sentiment Class", title_fontsize="11", loc="upper right")
plt.xticks(ticks=[0, 1, 2, 3], labels=["Tokopedia", "Shopee", "Lazada", "Bukalapak"], fontsize=11)
plt.tight_layout()
plt.show()

### 6. Validasi Manual (500 Sampel) & Pengujian Cohen's Kappa (Inter-Annotator Agreement)
Sesuai standar jurnal Scopus Q2/Q3, reliabilitas ground-truth diverifikasi secara formal menggunakan koefisien kesepakatan inter-annotator Cohen's Kappa ($\kappa$). Sebanyak 500 sampel acak terstratifikasi diuji antara anotasi rating dan validasi ahli manusia.

In [ ]:
import json
from sklearn.metrics import cohen_kappa_score

val_file = "../data/manual_validation_500.csv" if os.path.exists("../data/manual_validation_500.csv") else "data/manual_validation_500.csv"
val_df = pd.read_csv(val_file)
print(f"Manual Validation Dataset Loaded: {len(val_df)} rows")

# Hitung koefisien Cohen's Kappa
kappa = cohen_kappa_score(val_df["heuristic_label"], val_df["expert_validated_label"])
print(f"Cohen's Kappa Score: {kappa:.4f}")

kappa_json = "../paper_assets/cohen_kappa_validation.json" if os.path.exists("../paper_assets/cohen_kappa_validation.json") else "paper_assets/cohen_kappa_validation.json"
if os.path.exists(kappa_json):
    with open(kappa_json, "r", encoding="utf-8") as f:
        res = json.load(f)
    print(json.dumps(res, indent=2))